# Introduction
This notebook will contain different API-calling tests for our ATS systems.

In [2]:
# setup
import requests
import pandas as pd
from collections import defaultdict
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time

In [ ]:
# base class
class Scraper():
    '''Base scraper class.'''
    def __init__(self, csv_in: str, csv_out: str):
            '''Intialize a scraper instance.

            Params:
                csv_in: 
                    str - Filepath to a company-data csv for the corresponding ATS system.
    
                csv_out:
                    str - Filepath to desired results location.
            '''
            self.df = pd.read_csv(csv_in)
            self.csv_out = csv_out
            self.session = requests.Session()
            retries = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
            self.session.mount('https://', HTTPAdapter(max_retries=retries))

    def _truncate(self, s, n):
        return s if len(s) <= n else s[:n-1] + "…"

    def fetch():
        # fetch single companies job postings.
        raise NotImplementedError

    def get_jobs(
            self,
            limit: int= -1
        ) -> dict:
        '''Scrape platform for jobs using the defined inputs. Override function if not using slug 
        format.

        Params:
            limit:
                int=-1 - Limit of companies to check. -1 to search all possible companies.
            
        Returns:
            dict - A dictionary of results. results[company_identifier] = job list | error code
        '''

        # --- create variables
        self.INVALID = set() # invalid company identifiers
        results = defaultdict()
        i = 0
        # stats
        valid = 0
        has_jobs = 0
        total_jobs = 0

        # --- begin scrape
        start_time = time.time()
        print('id   slug                                               response           total/error')
        for slug in self.df.slug:
            if i == limit: break
            # - fetch
            res = self.fetch(slug) # should return a list of jobs, or an int with an error code

            # - validate
            if isinstance(res, list):
                valid += 1
                if len(res) > 0:
                    has_jobs += 1
                    total_jobs += len(res)
                print(f'{i:<4} {self._truncate(slug, 40):<40}           valid              total: {len(res)}')
            else:
                self.INVALID.add(slug)
                print(f'{i:<4} {self._truncate(slug, 40):<40}           invalid            error: {res}')

            # - update
            results[slug] = res
            i += 1

            # respect rate limit
            time.sleep(self.DELAY) # inherited from child
        end_time = time.time()

        self.SCRAPE_TIME = end_time - start_time


        # --- print stats
        print('-'*103)
        if i > 0:
            print(f'''scrape results:
    scraping time:        {end_time-start_time:.1f}s ({(end_time-start_time) / 60:.2f}m)
    scraped slugs:        {i}
    total jobs:           {total_jobs}
    -
    valid slugs:          {valid} ({(valid/i)*100:.1f}%)
    valid with jobs:      {has_jobs} 
    invalid:              {len(self.INVALID)} ({(len(self.INVALID)/i)*100:.1f}%)
''')
        else:
            print('no jobs scraped.')

        return results

    def process():
        # process jobs into a single-row format.
        raise NotImplementedError

    def run(
            self,
            limit: int=-1,
        ) -> pd.DataFrame:
        '''Will perform the entire scraping and processing loop. Saves results to *self.csv_out*.
        
        Params:
            limit:
                int=-1 - Limit of company slugs to check. -1 to search all possible slugs.

        Returns:
                    pd.DataFrame - DataFrame containing processed scrape information. DataFrame will contain
                    the following columns: [slug, company_name, title, board_id, url, date_posted, 
                    date_scraped]
        '''
        unprocessed = self.get_jobs(limit)
        processed = self.process(unprocessed)

        # save as csv
        print(f'saving results to "{self.csv_out}"...')
        processed.to_csv(self.csv_out)
        print(f'results saved to "{self.csv_out}".')
        return processed   

    def get_invalid(self) -> set[str]:
        return self.INVALID

    def get_scrape_time(self) -> tuple[int, int] | None:
        '''Returns scrape time in seconds, scrape time in minutes, or None.'''
        if self.SCRAPE_TIME:
            return self.SCRAPE_TIME, self.SCRAPE_TIME / 60
        else:
            return None

# Greenhouse.io

In [ ]:
class Greenhouse(Scraper):
    '''Greenhouse (*greenhouse.io*) scraper.
            
        ### Rate limit: 
            Greenhouse defines no rate limit on their job board API, however they define a 
            50-per-10-second limit on their audit log API, so to respect this we will use a 
            **.2s** delay.
    '''
    DELAY=0.2 
    
    def __init__(self, csv_in: str, csv_out: str):
        super().__init__(csv_in, csv_out)

    def fetch(self, slug: str) -> list | int:
        '''Will fetch a companies job listings using the Greenhouse Job Board API.
        
        Params:
            slug:
                str - The slug to request.
        
        Returns:
            list | str - A list of all jobs; if an error occurs, the int status code will be returned.
        '''
        try:
            # retrieve
            res = self.session.get(f'https://boards-api.greenhouse.io/v1/boards/{slug}/jobs')

            # detect error
            res.raise_for_status()

            # return jobs
            res_json = res.json()
            return res_json['jobs']
        except requests.exceptions.HTTPError as e:
            return res.status_code
        except Exception as e:
            print(f'------ unknown error occured for value {slug}: {e}')
            return -1

    def process(
            self,
            results: dict, 
        ) -> pd.DataFrame:
        '''Will process a given input of job results.
        
        Params:
            results: 
                dict - Unprocessed scrape results.
        
        Returns:
            pd.DataFrame - DataFrame containing processed scrape information. DataFrame will contain
            the following columns: [slug, company_name, title, board_id, url, date_posted, 
            date_scraped]
        '''
        df = pd.DataFrame(columns=['slug', 'company_name', 'title', 'board_id', 'url', 'date_posted', '_company_key'])

        # iterate through all slugs, and extract information from all valid job listings
        for slug in results.keys():
            jobs = results[slug]
            # process jobs
            if isinstance(jobs, list) and len(jobs) > 0:
                rows = []
                for job in jobs:
                    data = {
                        'slug':             slug,
                        'company_name':     str(job['company_name']).strip(),
                        'title':            str(job['title']).strip().lower(), 
                        'board_id':         job['id'],  
                        'url':              job['absolute_url'],
                        'date_posted':      pd.to_datetime(job['first_published'], utc=True), 
                        '_company_key':     str(job['company_name']).strip().lower()
                    }
                    rows.append(data)
                rows_df = pd.DataFrame(rows)
                df = pd.concat([df, rows_df], ignore_index=True)

        # add timestamp
        df['date_scraped'] = pd.Timestamp.now(tz='UTC')
        # remove duplicates
        #   duplicates may occur in instances where we have two slugs with different capitilzation 
        #   within our company slug file.
        df = df.drop_duplicates(subset=['_company_key', 'board_id'], keep='first')
        df = df.drop(columns=['_company_key'])
        return df

In [ ]:
greenhouse = Greenhouse('../data/companies/greenhouse.csv', '../data/jobs/greenhouse_jobs.csv')
_ = greenhouse.run(10)

id   slug                                               response           total/error
0    0verwatch                                          valid              total: 1
1    103644278                                          valid              total: 16
2    10alabs                                            valid              total: 16
3    10beauty                                           valid              total: 7
4    10xgenomics                                        invalid            error: 404
5    12jlkfsk                                           valid              total: 5
6    12twenty                                           valid              total: 4
7    143studiosinc                                      valid              total: 4
8    1456754456yhgbhfg                                  valid              total: 1
9    14technology                                       valid              total: 2
---------------------------------------------------------------------

# Smart Recruiters

In [ ]:
class SmartRecruiters(Scraper):
    '''SmartRecruiters (*smartrecruiters.com*) scraper.

    Rate limit:
    ---
        Smart recruiters lists an allowed 10 requests per second. We will use a **0.1s** delay.
    '''
    DELAY=0.1
    def __init__(self, csv_in: str, csv_out: str):
        super().__init__(csv_in, csv_out)

    def fetch(self, slug: str) -> list | int:
        '''Will fetch a companies job listings.
        
        Params:
            slug:
                str - The slug to request.
        
        Returns:
            list | str - A list of all jobs; if an error occurs, the int status code will be returned.
        '''
        # smartrecruiters will give a max of 100 jobs at once.
        # in the event a company has more than 100 jobs open, we must using a sliding window
        checked = 0
        total = None
        content = []
        res = None
        try:
            while total is None or checked < total:
                # respect rate limit when sliding window
                if checked > 0: time.sleep(0.1) 
    
                # retrieve
                res = self.session.get(
                    f'https://api.smartrecruiters.com/v1/companies/{slug}/postings',
                    params={
                        'limit': 100,
                        'offset': checked
                        }
                    )
    
                # check 
                res.raise_for_status()
    
                # process            
                res_json = res.json()
                if total is None: total = res_json['totalFound'] # update total
                content += res_json['content']
    
                # slide window
                checked += 100
            return content   
        except requests.exceptions.HTTPError as e:
            return res.status_code
        except Exception as e:
            print(f'------ unknown error occured for value {slug}: {e}')
            return -1

    def process(
            self,
            results: dict, 
        ) -> pd.DataFrame:
        '''Will process a given input of job results.
        
        Params:
            results: 
                dict - Unprocessed scrape results.
        
        Returns:
            pd.DataFrame - DataFrame containing processed scrape information. DataFrame will contain
            the following columns: [slug, company_name, title, board_id, url, date_posted, 
            date_scraped]
        '''
        df = pd.DataFrame(columns=['slug', 'company_name', 'title', 'board_id', 'url', 'date_posted', '_company_key'])
        
        # iterate through all slugs, and extract information from all valid job listings
        for slug in results.keys():
            jobs = results[slug]
            # process jobs
            if isinstance(jobs, list) and len(jobs) > 0:
                rows = []
                for job in jobs:
                    data = {
                        'slug':             slug,
                        'company_name':     str(job['company']['name']).strip(),
                        'title':            str(job['name']).strip().lower(), 
                        'board_id':         job['id'],  
                        'url':              job['ref'],
                        'date_posted':      pd.to_datetime(job['releasedDate'], utc=True), 
                        '_company_key':     str(job['company']['name']).strip().lower()
                    }
                    rows.append(data)
                rows_df = pd.DataFrame(rows)
                df = pd.concat([df, rows_df], ignore_index=True)

        # add timestamp
        df['date_scraped'] = pd.Timestamp.now(tz='UTC')
        # remove duplicates
        #   duplicates may occur in instances where we have two slugs with different capitilzation 
        #   within our company slug file.
        df = df.drop_duplicates(subset=['_company_key', 'board_id'], keep='first')
        df = df.drop(columns=['_company_key'])
        return df


In [ ]:
smart_recruiters = SmartRecruiters('../data/companies/smartrecruiters.csv','../data/jobs/smartrecruiters_jobs.csv')
_ = smart_recruiters.run(10)

id   slug                                               response           total/error
0    10xValuePartnersGmbH                               valid              total: 5
1    1839Labs                                           valid              total: 0
2    1HUDDLE                                            valid              total: 9
3    1Huddle                                            valid              total: 9
4    1perTInentInc                                      valid              total: 10
5    2NTELEKOMUNIKACEAs                                 valid              total: 12
6    336ExpressLLC                                      valid              total: 2
7    3HPartners                                         valid              total: 6
8    4PsCorporation                                     valid              total: 0
9    A2Design                                           valid              total: 3
-----------------------------------------------------------------------

# Ashby

In [ ]:
class Ashby(Scraper):
    '''Ashby (*ashbyhq.com*) scraper.
    
    Rate limit
    ---
    Ashby does not define a rate limit on their API website, we will use a **.2s** delay.
    '''
    DELAY=0.2
    def __init__(self, csv_in: str, csv_out: str):
        super().__init__(csv_in, csv_out)

    def fetch(self, slug: str) -> list | int:
        '''Will fetch a companies job listings.
        
        Params:
            slug:
                str - The slug to request.
        
        Returns:
            list | str - A list of all jobs; if an error occurs, the int status code will be returned.
        '''
        res = None
        try:
            # retrieve
            res = self.session.get(f'https://api.ashbyhq.com/posting-api/job-board/{slug}')

            # detect error
            res.raise_for_status()

            # return jobs
            res_json = res.json()
            return res_json['jobs']
        except requests.exceptions.HTTPError as e:
            return res.status_code
        except Exception as e:
            print(f'------ unknown error occured for value {slug}: {e}')
            return -1

    def process(
            self,
            results: dict, 
        ) -> pd.DataFrame:
        '''Will process a given input of job results.
        
        Params:
            results: 
                dict - Unprocessed scrape results.
        
        Returns:
            pd.DataFrame - DataFrame containing processed scrape information. DataFrame will contain
            the following columns: [slug, company_name, title, board_id, url, date_posted, 
            date_scraped]
        '''
        df = pd.DataFrame(columns=['slug', 'company_name', 'title', 'board_id', 'url', 'date_posted', '_company_key'])

        # iterate through all slugs, and extract information from all valid job listings
        for slug in results.keys():
            jobs = results[slug]
            # process jobs
            if isinstance(jobs, list) and len(jobs) > 0:
                rows = []
                for job in jobs:
                    data = {
                        'slug':             slug,
                        'company_name':     None, # not listed in ashby job api results
                        'title':            str(job['title']).strip().lower(), 
                        'board_id':         job['id'],  
                        'url':              job['jobUrl'],
                        'date_posted':      pd.to_datetime(job['publishedAt'], utc=True), 
                        '_company_key':     str(slug).lower()
                    }
                    rows.append(data)
                rows_df = pd.DataFrame(rows)
                df = pd.concat([df, rows_df], ignore_index=True)

        # add timestamp
        df['date_scraped'] = pd.Timestamp.now(tz='UTC')
        # remove duplicates
        #   duplicates may occur in instances where we have two slugs with different capitilzation 
        #   within our company slug file.
        df = df.drop_duplicates(subset=['_company_key', 'board_id'], keep='first')
        df = df.drop(columns=['_company_key'])
        return df

In [ ]:
ashby = Ashby('../data/companies/ashby.csv', '../data/jobs/asbhy_jobs.csv')
_ = ashby.run(10)

id   slug                                               response           total/error
0    01c                                                invalid            error: 404
1    0g                                                 valid              total: 8
2    0x                                                 valid              total: 1
3    10xteam                                            valid              total: 133
4    11x                                                invalid            error: 404
5    1bios                                              valid              total: 2
6    1mind                                              valid              total: 6
7    1password                                          valid              total: 64
8    1st                                                invalid            error: 404
9    1x                                                 valid              total: 92
---------------------------------------------------------------

# Breezy

In [ ]:
class Breezy(Scraper):
    '''Breezy (*breezy.hr*) scraper.
    
    Rate limit
    ---
    Breezy defines an 100-requests/min rate limit, we will use a **0.6s** delay.
    '''
    DELAY=0.6
    def __init__(self, csv_in: str, csv_out: str):
        super().__init__(csv_in, csv_out)

    def fetch(self, slug: str) -> list | int:
        '''Will fetch a companies job listings.
        
        Params:
            slug:
                str - The slug to request.
        
        Returns:
            list | str - A list of all jobs; if an error occurs, the int status code will be returned.
        '''
        res = None
        try:
            # retrieve
            res = self.session.get(f'https://{slug}.breezy.hr/json')
            
            # detect error
            res.raise_for_status()

            # return jobs
            res_json = res.json()
            return res_json
        except requests.exceptions.HTTPError as e:
            return res.status_code
        except Exception as e:
            print(f'------ unknown error occured for value {slug}: {e}')
            return -1

    def process(
            self,
            results: dict, 
        ) -> pd.DataFrame:
        '''Will process a given input of job results.
        
        Params:
            results: 
                dict - Unprocessed scrape results.
        
        Returns:
            pd.DataFrame - DataFrame containing processed scrape information. DataFrame will contain
            the following columns: [slug, company_name, title, board_id, url, date_posted, 
            date_scraped]
        '''
        df = pd.DataFrame(columns=['slug', 'company_name', 'title', 'board_id', 'url', 'date_posted', '_company_key'])

        # iterate through all slugs, and extract information from all valid job listings
        for slug in results.keys():
            jobs = results[slug]
            # process jobs
            if isinstance(jobs, list) and len(jobs) > 0:
                rows = []
                for job in jobs:
                    data = {
                        'slug':             slug,
                        'company_name':     str(job['company']['name']).strip(),
                        'title':            str(job['name']).strip().lower(), 
                        'board_id':         job['id'],  
                        'url':              job['url'],
                        'date_posted':      pd.to_datetime(job['published_date'], utc=True), 
                        '_company_key':     str(job['company']['name']).strip().lower()
                    }
                    rows.append(data)
                rows_df = pd.DataFrame(rows)
                df = pd.concat([df, rows_df], ignore_index=True)

        # add timestamp
        df['date_scraped'] = pd.Timestamp.now(tz='UTC')
        # remove duplicates
        #   duplicates may occur in instances where we have two slugs with different capitilzation 
        #   within our company slug file.
        df = df.drop_duplicates(subset=['_company_key', 'board_id'], keep='first')
        df = df.drop(columns=['_company_key'])
        return df

In [24]:
breezy = Breezy('../data/companies/breezy.csv', '../data/jobs/breezy_jobs.csv')
_ = breezy.run(10)

id   slug                                               response           total/error
0    021-strategic                                      valid              total: 3
1    1-grid                                             valid              total: 1
2    1001                                               valid              total: 23
3    10percent                                          invalid            error: 404
4    11onze                                             valid              total: 0
5    11sigma                                            valid              total: 0
6    1234s                                              valid              total: 1
7    1440-foods-manufacturing                           valid              total: 28
8    1aes                                               valid              total: 4
9    1kx                                                valid              total: 0
---------------------------------------------------------------------

In [3]:
res = requests.get('https://021-strategic.breezy.hr/json')

In [ ]:

['slug', 'company_name', 'title', 'board_id', 'url', 'date_posted', '_company_key']

'id', 'friendly_id', 'name', 'url', 'published_date', 'type', 'location', 'department', 'salary', 'company', 'locations'

res.json()[0]

[{'id': '6edb824628c1',
  'friendly_id': '6edb824628c1-grants-specialist',
  'name': 'Grants Specialist',
  'url': 'https://021-strategic.breezy.hr/p/6edb824628c1-grants-specialist',
  'published_date': '2026-09-23T12:38:31.603Z',
  'type': {'id': 'fullTime', 'name': 'Full-Time'},
  'location': {'country': {'name': 'Canada', 'id': 'CA'},
   'state': {'id': 'ON', 'name': 'Ontario'},
   'primary': True,
   'is_remote': True,
   'remote_details': {'value': 'remote',
    'label': 'Fully remote, no location restrictions'},
   'name': 'Canada'},
  'department': None,
  'salary': '$75,000 – $80,000 / year',
  'company': {'name': 'RightBlue Labs Inc',
   'logo_url': None,
   'friendly_id': '021-strategic',
   'isMultipleLocationsEnabled': True},
  'locations': [{'country': {'name': 'Canada', 'id': 'CA'},
    'state': {'id': 'ON', 'name': 'Ontario'},
    'primary': True,
    'is_remote': True,
    'remote_details': {'value': 'remote',
     'label': 'Fully remote, no location restrictions'},
   

In [11]:
(60/ 100)

0.6